# Trie + SONAR-Gated LogitsProcessor: Training-Free Contextual Biasing for Whisper, Evaluated on Real Audio

**Prerequisite:** `whisper_to_sonar_alignment_experiment.ipynb` passed (top-1 ≈ 100% held-out retrieval,
best layer = 4). This notebook consumes its artifact `whisper_to_sonar_W.pt` and builds the actual
inference pipeline the original guide sketched — then measures whether it helps on **real speech**.

---

## What we are testing (three conditions)

| Condition | Decoding | What it tells us |
|---|---|---|
| **A — Baseline** | plain Whisper beam search | reference WER / how often rare words are lost |
| **B — Trie-only** | +δ logit boost for any token continuing a biasing-list word | the classic, proven shallow-fusion baseline |
| **C — Trie + SONAR gate** | boost = δ + λ·cos(projected decoder state, word's SONAR embedding) | **does semantic gating add value over B?** |

## The key experimental trick: distractors

The biasing list is 50% **targets** (rare words that really occur in the eval audio) and 50%
**distractors** (equally rare words mined from *other* utterances, guaranteed absent from the eval
references). A dumb booster (B) pushes both equally — raising δ buys recall but hallucinates
distractors into transcripts. The semantic gate (C) should assign distractors low cosine similarity
against the utterance's evolving decoder state and withhold their boost. **The C-vs-B false-alarm gap
at matched recall is the entire justification for the SONAR machinery.**

## Metrics (computed from jiwer word alignments)

- **WER** — overall word error rate (biasing must not degrade it).
- **B-WER / U-WER** — error rate restricted to biased / unbiased reference words (Le et al.-style):
  substitutions+deletions attributed by the reference word, insertions by the hypothesis word.
- **Hotword recall** — fraction of target-word occurrences in the reference that appear aligned-correct
  in the hypothesis. The number biasing exists to raise.
- **False alarms** — biasing-list words inserted or substituted into the hypothesis where the reference
  has no such word. The number biasing must not raise.

## Pipeline architecture built here

```
                        ┌──────────────────────────────────────────────┐
 audio ──► encoder ──►  │  Whisper beam search (HF generate)           │──► transcript
                        │    ▲ logits            │ hidden states       │
                        │    │                   ▼ (hook, layer 4)     │
                        │  TrieSonarBiasProcessor                      │
                        │    • walk trie over each beam's token suffix │
                        │    • pool text states → @W → cos vs SONAR    │
                        │    • boost = δ + λ·max(0, cos)               │
                        └──────────────────────────────────────────────┘
```

**Runtime note.** Condition C needs full-sequence decoder states each step, so it runs with the KV
cache disabled (honest but slow — a production version would maintain per-beam pooled states
incrementally; see Limitations). Defaults below auto-shrink on CPU; on a CUDA machine expect
~10–20 min total.


---
## Section 0.1 — Dependencies

Beyond notebook 1's stack we add:
- `jiwer` — WER with word-level alignments (we need the alignment chunks, not just the score, to
  compute B-WER/U-WER/recall/false-alarms correctly).
- `soundfile` / `librosa` — audio decoding fallbacks for `datasets`' Audio feature across versions.


In [ ]:
# Run once, then restart the kernel.
# %pip install -U torch transformers datasets scikit-learn scipy pandas matplotlib jiwer soundfile librosa
# %pip install sonar-space          # SONAR — required (embeds the biasing words)


## Section 0.2 — Imports, seed, device

Same conventions as notebook 1. `itertools.islice` is used to take a bounded number of utterances from
a *streaming* dataset (so we never download all of LibriSpeech).


In [ ]:
import os, math, random, re, json, itertools
from collections import Counter
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("Using device:", DEVICE)


## Section 0.3 — Configuration

- `best_layer = 4` — the winner from notebook 1's layer sweep. **Indexing note:** `decoder_hidden_states[4]`
  is the output of decoder *block* 4, which is the module `model.decoder.layers[3]` — the hook in
  Section 5 uses `best_layer − 1` for exactly this reason.
- `w_path` — the alignment matrix from notebook 1. If missing, Section 1 refits a quick replacement so
  this notebook stands alone.
- `n_dev` / `n_eval` — dev utterances tune (δ, λ); eval utterances produce the final table. Never mixed.
- `n_targets` / `n_distractors` — biasing-list composition (the 50/50 split described above).
- `delta_grid` / `lambda_grid` — the sweep. δ is per-token log-prob bonus; typical shallow-fusion values
  are 0.5–4. λ scales the cosine (~0.2–0.5 for true context per notebook 1), so λ=4 adds ≲ 2 logits.
- `FAST` — auto-True on CPU: shrinks counts/beams so the notebook finishes in reasonable time; set it
  False manually on a GPU box for the full run.


In [ ]:
FAST = (DEVICE == "cpu")

CONFIG = {
    "whisper_model": "openai/whisper-base",
    "best_layer": 4,
    "w_path": "whisper_to_sonar_W.pt",
    "n_dev":   8 if FAST else 15,
    "n_eval": 20 if FAST else 50,
    "n_extra_texts": 300,
    "n_targets": 25 if FAST else 50,
    "n_distractors": 25 if FAST else 50,
    "beams": 3 if FAST else 5,
    "delta_grid":  [1.0, 2.0, 3.0],
    "lambda_grid": [2.0, 4.0, 8.0],
    "min_word_len": 5,
    "max_doc_freq": 3,
}
print("FAST mode:", FAST)
CONFIG


---
## Section 1 — Load frozen Whisper and the alignment matrix `W`

Line-by-line:
1. Load model/processor exactly as notebook 1; `.eval()` + `no_grad` everywhere — nothing trains.
2. `set_prefix_tokens` pins the decoding prompt to `<|sot|><|en|><|transcribe|><|notimestamps|>`;
   `N_PREFIX` lets the bias processor skip these positions both when walking the trie (they are not
   text) and when pooling hidden states.
3. Try to load `whisper_to_sonar_W.pt` (shape `[512, 1024]`). If absent, the **fallback cell below**
  refits a compact version (800 WikiText sentences, layer 4, Ridge α=10 — a miniature of notebook 1)
  so results stay comparable. If you have the real artifact from the machine that ran notebook 1,
  copy it next to this notebook instead — a matrix fitted on 1,600 sentences is strictly better.


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers import LogitsProcessor, LogitsProcessorList

processor = WhisperProcessor.from_pretrained(CONFIG["whisper_model"])
whisper = (WhisperForConditionalGeneration
           .from_pretrained(CONFIG["whisper_model"]).to(DEVICE).eval())
tok = processor.tokenizer
tok.set_prefix_tokens(language="english", task="transcribe")
N_PREFIX = len(tok.prefix_tokens)
D_MODEL = whisper.config.d_model
print(f"Whisper loaded: d_model={D_MODEL}, prefix={tok.convert_ids_to_tokens(tok.prefix_tokens)}")

W = None
if os.path.exists(CONFIG["w_path"]):
    W = torch.load(CONFIG["w_path"], map_location="cpu").float()
    print(f"Loaded alignment matrix: {tuple(W.shape)} from {CONFIG['w_path']}")
else:
    print("!! No W found — run the fallback refit cell below.")


### Fallback: quick refit of `W` (skipped automatically if the artifact was found)

A condensed replay of notebook 1: silence-conditioned pooled states at layer 4 for 800 WikiText
sentences → SONAR embeddings → Ridge(α=10). ~5 minutes. Every line mirrors notebook 1's Section 2/5;
see there for the full rationale.


In [ ]:
if W is None:
    from datasets import load_dataset
    from sklearn.linear_model import Ridge
    from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline

    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    sents = []
    for line in ds["text"]:
        line = line.strip().replace(" @-@ ", "-").replace(" @,@ ", ",").replace(" @.@ ", ".")
        if not line or line.startswith("="):
            continue
        for s in re.split(r"(?<=[.!?]) +", line):
            if 40 <= len(s.strip()) <= 200:
                sents.append(s.strip())
    sents = list(dict.fromkeys(sents))
    random.Random(SEED).shuffle(sents)
    sents = sents[:800]

    sil = processor.feature_extractor(np.zeros(16000 * 30, dtype=np.float32),
                                      sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
    with torch.no_grad():
        SIL_ENC_FIT = whisper.get_encoder()(sil).last_hidden_state
        X = []
        for i in range(0, len(sents), 16):
            enc = tok(sents[i:i+16], return_tensors="pt", padding=True)
            ids, attn = enc.input_ids.to(DEVICE), enc.attention_mask.to(DEVICE)
            out = whisper(encoder_outputs=(SIL_ENC_FIT.expand(ids.shape[0], -1, -1),),
                          decoder_input_ids=ids, output_hidden_states=True, return_dict=True)
            hs = out.decoder_hidden_states[CONFIG["best_layer"]]
            m = attn.clone(); m[:, :N_PREFIX] = 0
            m.scatter_(1, attn.sum(1, keepdim=True) - 1, 0)
            m = m.unsqueeze(-1).float()
            X.append(F.normalize((hs * m).sum(1) / m.sum(1).clamp(min=1), dim=-1).cpu())
    X = torch.cat(X).numpy()

    t2v = TextToEmbeddingModelPipeline(encoder="text_sonar_basic_encoder",
                                       tokenizer="text_sonar_basic_encoder",
                                       device=torch.device("cpu"))
    Y = F.normalize(t2v.predict(sents, source_lang="eng_Latn", batch_size=64).float(), dim=-1).numpy()

    W = torch.from_numpy(Ridge(alpha=10.0, fit_intercept=False).fit(X, Y).coef_.T).float()
    torch.save(W, CONFIG["w_path"])
    print(f"Refit W: {tuple(W.shape)} → saved to {CONFIG['w_path']}")
else:
    print("W already loaded — refit skipped.")
W_DEV = W.to(DEVICE)


---
## Section 2 — SONAR (embeds the biasing words)

At inference time SONAR is used **once per biasing list**, never per audio frame — it embeds each
biasing word into the 1024-d space that `W` projects into. We embed the bare lowercase word; SONAR is
a sentence encoder, but single words produce usable embeddings (a templating variant like
"a recording about {word}" is an easy later experiment).


In [ ]:
from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline

t2vec = TextToEmbeddingModelPipeline(encoder="text_sonar_basic_encoder",
                                     tokenizer="text_sonar_basic_encoder",
                                     device=torch.device("cpu"))

def sonar_embed(words):
    """list[str] -> unit-norm tensor [len(words), 1024] on DEVICE."""
    with torch.no_grad():
        e = t2vec.predict(list(words), source_lang="eng_Latn", batch_size=64).float()
    return F.normalize(e, dim=-1).to(DEVICE)

print("SONAR ready.")


---
## Section 3 — Real audio: LibriSpeech (streaming)

Line-by-line:
1. `streaming=True` — we iterate the corpus lazily and keep only what we take with `islice`; nothing
   close to the full 30 GB is downloaded. If the main dataset is unreachable, we fall back to the tiny
   73-utterance `librispeech_asr_dummy` (enough to demo the pipeline, too small for solid WER — the
   printout tells you which one you got).
2. `cast_column(..., Audio(sampling_rate=16000))` — guarantees 16 kHz mono, Whisper's expected input.
3. The stream is split **positionally**: first `n_dev` utterances → dev (tuning), next `n_eval` → eval
   (final numbers), next `n_extra_texts` → *text only*, used to mine distractor words that are
   guaranteed absent from dev/eval references.
4. `get_audio()` absorbs the API differences between `datasets` versions (dict with `array`, torchcodec
   `AudioDecoder`, or a path for `soundfile`) and always returns a float32 numpy vector.


In [ ]:
from datasets import load_dataset, Audio

try:
    stream = load_dataset("openslr/librispeech_asr", "clean", split="validation", streaming=True)
    DATA_NAME = "LibriSpeech validation.clean (streaming)"
except Exception as e:
    print("librispeech_asr unavailable → dummy fallback:", repr(e)[:120])
    stream = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean",
                          split="validation", streaming=True)
    DATA_NAME = "librispeech_asr_dummy (73 utts — demo only)"
stream = stream.cast_column("audio", Audio(sampling_rate=16000))

def get_audio(sample):
    a = sample["audio"]
    if isinstance(a, dict) and a.get("array") is not None:
        return np.asarray(a["array"], dtype=np.float32)
    if hasattr(a, "get_all_samples"):                       # torchcodec AudioDecoder
        return a.get_all_samples().data.numpy().astype(np.float32).flatten()
    import soundfile as sf
    return sf.read(a["path"], dtype="float32")[0]

it = iter(stream)
dev_set  = list(itertools.islice(it, CONFIG["n_dev"]))
eval_set = list(itertools.islice(it, CONFIG["n_eval"]))
extra_texts = [s["text"] for s in itertools.islice(it, CONFIG["n_extra_texts"])]

print(f"{DATA_NAME}\ndev={len(dev_set)}  eval={len(eval_set)}  extra_texts={len(extra_texts)}")
print("sample ref:", eval_set[0]["text"][:90])
print("sample audio:", get_audio(eval_set[0]).shape)


---
## Section 4 — Building the biasing list (targets + distractors)

Line-by-line:
1. `norm()` — one normalization used *everywhere* (mining, WER, recall): lowercase, strip punctuation
   except in-word apostrophes, collapse whitespace. LibriSpeech refs are uppercase/unpunctuated;
   Whisper outputs cased+punctuated text — without a shared normalizer every metric would be wrong.
2. **Targets** = words in the *eval* references that are ≥5 chars, alphabetic, and appear in ≤3
   documents across everything we've seen — i.e. genuinely rare words Whisper plausibly misses.
   Rarity is measured by document frequency over dev+eval+extra (~370 transcripts).
3. **Distractors** = words passing the same rarity filter but drawn from `extra_texts` and required to
   be **absent from every dev/eval reference** — they can only ever hurt, which is the point.
4. Both lists are truncated to configured sizes; `is_target`/`HOTSET` power the metrics later.
5. `sonar_embed` produces `E_bias [H, 1024]`, aligned index-for-index with `bias_words`.


In [ ]:
def norm(text):
    text = text.lower()
    text = re.sub(r"[^a-z' ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def doc_freq(texts):
    c = Counter()
    for t in texts:
        c.update(set(norm(t).split()))
    return c

dev_refs  = [s["text"] for s in dev_set]
eval_refs = [s["text"] for s in eval_set]
df = doc_freq(dev_refs + eval_refs + extra_texts)

ok = lambda w: len(w) >= CONFIG["min_word_len"] and w.isalpha() and df[w] <= CONFIG["max_doc_freq"]

eval_vocab = [w for t in eval_refs for w in norm(t).split()]
targets = list(dict.fromkeys(w for w in eval_vocab if ok(w)))[:CONFIG["n_targets"]]

seen = set(w for t in dev_refs + eval_refs for w in norm(t).split())
extra_vocab = [w for t in extra_texts for w in norm(t).split()]
distractors = list(dict.fromkeys(
    w for w in extra_vocab if ok(w) and w not in seen))[:CONFIG["n_distractors"]]

bias_words = targets + distractors
is_target = [True] * len(targets) + [False] * len(distractors)
HOTSET = set(bias_words)
TARGETSET = set(targets)

print(f"targets({len(targets)}):", targets[:8], "...")
print(f"distractors({len(distractors)}):", distractors[:8], "...")

E_bias = sonar_embed(bias_words)
print("SONAR bias embeddings:", tuple(E_bias.shape))


---
## Section 5 — The biasing trie (why standard, and the casing trap)

As established in the guide: a **standard trie** (one BPE token per edge) matches beam search's
one-token-per-step rhythm, which is why we don't use a radix tree.

**The casing/space trap** — the practical detail most implementations get wrong: Whisper's BPE is
context-sensitive. `"whittier"`, `"Whittier"`, `" whittier"` and `" Whittier"` tokenize to *different
ID sequences*, and mid-sentence words carry the leading space inside the first token. So each biasing
word is inserted under **four surface forms** (±capitalization × ±leading space), all mapping to the
same hotword id (same SONAR embedding, same metrics identity).

Data structure, line-by-line:
- `TrieNode.children` — dict `token_id → TrieNode` (edge = exactly one BPE token).
- `TrieNode.hids` — the set of hotword ids whose paths pass through this node; when the processor
  boosts the edge *into* a node it looks up `hids` to fetch the relevant SONAR similarities.
- `insert()` walks/creates the path and tags every node with the hotword id.
- `walk(seq)` returns the node reached by consuming `seq` from the root, or `None` — used by the
  processor to test each beam's token *suffix*.
- `max_depth` bounds how long a suffix can possibly matter, keeping the per-step cost O(depth²) per
  beam with tiny constants.
The unit test at the bottom must print the same node for both casings and `None` for a non-member.


In [ ]:
class TrieNode:
    __slots__ = ("children", "hids")
    def __init__(self):
        self.children = {}
        self.hids = set()

class BiasTrie:
    def __init__(self):
        self.root = TrieNode()
        self.max_depth = 0

    def insert(self, token_ids, hid):
        node = self.root
        for t in token_ids:
            node = node.children.setdefault(t, TrieNode())
            node.hids.add(hid)
        self.max_depth = max(self.max_depth, len(token_ids))

    def walk(self, seq):
        node = self.root
        for t in seq:
            node = node.children.get(t)
            if node is None:
                return None
        return node

trie = BiasTrie()
n_paths = 0
for hid, w in enumerate(bias_words):
    for form in {w, w.capitalize()}:
        for surface in (form, " " + form):
            ids = tok.encode(surface, add_special_tokens=False)
            if ids:
                trie.insert(ids, hid)
                n_paths += 1
print(f"Trie built: {len(bias_words)} words → {n_paths} surface paths, max_depth={trie.max_depth}")

probe = bias_words[0]
a = trie.walk(tok.encode(" " + probe, add_special_tokens=False))
b = trie.walk(tok.encode(" " + probe.capitalize(), add_special_tokens=False))
c = trie.walk(tok.encode(" zzqx", add_special_tokens=False))
print(f"unit test → '{probe}': node={a is not None}, capitalized={b is not None}, junk={c is None}")
assert a is not None and b is not None and c is None


---
## Section 6 — Capturing decoder states mid-generation (the hook)

HF's `LogitsProcessor` API hands us token ids and logits — **not** hidden states. The bridge is a
forward hook on the decoder layer whose output is `decoder_hidden_states[best_layer]`; per the
indexing note in Section 0.3 that module is `whisper.model.decoder.layers[best_layer − 1]`.

Line-by-line:
- `StateCapture.__call__(module, inputs, output)` — PyTorch calls this after every forward of the
  hooked module; decoder layers return a tuple whose element 0 is the hidden-state tensor
  `[num_beams, seq_len, 512]`, which we stash (detached — no autograd graph).
- Because condition C generates with `use_cache=False`, the decoder reprocesses the **full sequence
  every step**, so `seq_len` is the whole hypothesis so far — exactly what prefix pooling needs, and
  automatically consistent after beam reordering (nothing is accumulated across steps, so there is no
  stale per-beam state to invalidate). That correctness guarantee is what the slowdown buys; the
  incremental-cache version is discussed in Limitations.
- The hook stays registered permanently; conditions A/B simply never read `capture.hidden`.


In [ ]:
class StateCapture:
    def __init__(self):
        self.hidden = None
    def __call__(self, module, inputs, output):
        h = output[0] if isinstance(output, tuple) else output
        self.hidden = h.detach()          # [beams, seq_len, d_model]

capture = StateCapture()
hook_handle = whisper.model.decoder.layers[CONFIG["best_layer"] - 1].register_forward_hook(capture)
print(f"Hook registered on decoder block {CONFIG['best_layer']} "
      f"(module index {CONFIG['best_layer'] - 1}).")


---
## Section 7 — `TrieSonarBiasProcessor`: the heart of the pipeline

Called by `generate()` **once per decoding step**, receiving every live beam's tokens
(`input_ids [beams, cur_len]`) and next-token logits (`scores [beams, vocab]`). It must return
modified scores. What each part does:

**Semantic pass (once per step, all beams vectorized):**
1. Guard: only when `lam > 0`, a captured state exists, and at least one text token has been emitted
   (`seq_len > N_PREFIX`) — at the very first step there is nothing to pool, so C gracefully degrades
   to trie-only behavior.
2. Pool positions `N_PREFIX:` (the emitted text), L2-normalize, project through `W`, renormalize —
   *identical* math to notebook 1's prefix probe, which is precisely why that probe was the right
   pre-test.
3. `sims = proj @ E_bias.T → [beams, H]` — every beam's cosine against every biasing word, one matmul.

**Trie pass (per beam):**
4. `gen` = the beam's tokens after the 4-token prompt — the text Whisper has committed to.
5. Try every suffix of `gen` up to `max_depth − 1` tokens **including the empty suffix** (empty suffix
   = root = "a new biasing word could start at the next token"; the leading-space surface forms ensure
   this only fires at word boundaries).
6. Each valid suffix node contributes its children as boostable next tokens with
   `bonus = δ + λ·max(0, max sims over the child's hids)`: δ is the unconditional trie boost (recall
   floor even when the semantic estimate is noisy — remember the 29% top-1 at 25% prefix), the λ term
   is the semantic gate that separates targets from distractors. `max(0,·)` never *penalizes* — this
   processor only adds.
7. Overlapping suffix matches keep the **max** bonus per token (never stack), then one indexed add
   into `scores`.

`stats` counts boosted steps so the smoke test can prove the processor actually fired.


In [ ]:
class TrieSonarBiasProcessor(LogitsProcessor):
    def __init__(self, trie, E_bias, W, capture, delta, lam, n_prefix):
        self.trie, self.E = trie, E_bias
        self.W = W.to(E_bias.device)
        self.capture = capture
        self.delta, self.lam = float(delta), float(lam)
        self.n_prefix = n_prefix
        self.stats = {"steps": 0, "boosted_steps": 0}

    def __call__(self, input_ids, scores):
        self.stats["steps"] += 1
        beams = input_ids.shape[0]

        sims = None
        if self.lam > 0 and self.capture.hidden is not None:
            hs = self.capture.hidden
            if hs.shape[0] == beams and hs.shape[1] > self.n_prefix:
                pooled = F.normalize(hs[:, self.n_prefix:, :].mean(dim=1), dim=-1)
                proj = F.normalize(pooled.float() @ self.W, dim=-1)
                sims = (proj @ self.E.T).cpu()               # [beams, H]

        for b in range(beams):
            gen = input_ids[b, self.n_prefix:].tolist()
            bonus = {}
            lo = max(0, len(gen) - self.trie.max_depth + 1)
            for start in range(lo, len(gen) + 1):            # includes empty suffix → root
                node = self.trie.walk(gen[start:])
                if node is None:
                    continue
                for t, child in node.children.items():
                    val = self.delta
                    if sims is not None and child.hids:
                        s = max(sims[b, h].item() for h in child.hids)
                        val += self.lam * max(0.0, s)
                    if val > bonus.get(t, 0.0):
                        bonus[t] = val
            if bonus:
                self.stats["boosted_steps"] += 1
                idx = torch.tensor(list(bonus.keys()), device=scores.device)
                val = torch.tensor(list(bonus.values()), device=scores.device,
                                   dtype=scores.dtype)
                scores[b].index_add_(0, idx, val)
        return scores

print("TrieSonarBiasProcessor defined.")


---
## Section 7b — V2 processor: first-token asymmetry + failure-arc revocation

The error-driven evaluation exposed two mechanical flaws in the V1 processor: the **word-start
distortion field** (first tokens of *all* list words get +δ at every word boundary) and **orphaned
fragments** (a beam dragged partway into a keyword keeps its earned bonus after the acoustics
disagree — "HUSBANDMEN" → "Wea Husspundman"). V2 fixes both with one reformulation.

### Potential-based boost shaping

Define the bonus a hypothesis *deserves*:

```
Φ(hyp) = Σ over completed keyword occurrences [ η·δ + δ·(L−1) + λ·max(0, cos) ]
       + E_partial,   where E_partial = η·δ + δ·(d−1) for the current in-progress
                      match of depth d ≥ 1 (0 otherwise)
```

and give every candidate token the **marginal** score `Φ(hyp + token) − Φ(hyp)`. Every desired
behavior is now a theorem, not a special case:

| Event (candidate token…) | Marginal bonus |
|---|---|
| starts a keyword (depth 0 → 1) | **η·δ** — the asymmetry knob; η < 1 shrinks the word-start floor |
| continues a match (d → d+1) | **δ** — full continuation reward, keeps partial words alive in the beam |
| **completes** a keyword of length L | **δ + λ·max(0, cos)** — the semantic term lands exactly once, at completion, when the pooled state estimates relevance best; the chain is then *locked* (never revoked) |
| **abandons** a partial match (any non-continuing token, incl. EOS) | **−E_partial** — full revocation; an abandoned excursion nets exactly zero |

Total shaping any finished hypothesis has received = Σ over *completed* keywords only — the
Aho-Corasick-style guarantee, obtained without explicit failure arcs.

### Implementation notes (why the code looks the way it does)

- **Stateless, like V1:** each step rescans the beam's generated tokens with a greedy longest-suffix
  automaton (`scan_trie`), so beam reordering can never desynchronize state. `scan_trie` also
  returns completed occurrences (greedy, non-overlapping: state resets to root after a completion —
  keywords that are prefixes of longer keywords lose the longer continuation; a documented,
  second-order simplification).
- **Revocation as a broadcast:** when a partial of credit `E` is active, the code does
  `scores[b] −= E` over the whole row, then adds `E + marginal` back onto continuing tokens. A
  uniform within-row shift doesn't change that beam's internal ranking, but it lowers the beam's
  *cumulative* score against other beams unless the match is continued — precisely the intent, and
  it covers EOS-while-mid-word for free.
- **Candidate-level fallback approximation:** for a *leaving* token we credit at most a fresh
  depth-1 start (root children), not deeper cross-boundary suffix matches. The next step's full
  rescan self-corrects any underestimate — everything is recomputed from the tokens.
- **Semantic term moved to completion:** V1 spent λ·cos on every continuation token (unrevocable
  spend on words that may never complete); V2 spends it once, on realized words. Trade-off: mid-word
  survival now rests on δ alone — if sweeps show rare targets dying mid-word, a small per-token
  semantic advance (revocable, at η_sem·λ·cos) is the natural V3 extension.
- `TrieV2` is a self-contained rebuild (V1's `TrieNode.__slots__` can't carry the needed `end_hids`
  terminal markers).

Constructor is drop-in compatible with V1 plus `eta` (default 0.25): swapping classes in the harness
is a one-line change.


In [ ]:
class TrieV2:
    """BPE trie with terminal markers: end_hids = keyword ids that END at this node."""
    class Node:
        __slots__ = ("children", "hids", "end_hids")
        def __init__(self):
            self.children = {}
            self.hids = set()
            self.end_hids = set()

    def __init__(self):
        self.root = self.Node()
        self.max_depth = 0

    def insert(self, token_ids, hid):
        node = self.root
        for t in token_ids:
            node = node.children.setdefault(t, TrieV2.Node())
            node.hids.add(hid)
        node.end_hids.add(hid)
        self.max_depth = max(self.max_depth, len(token_ids))


def scan_trie(trie, gen):
    """Greedy longest-suffix automaton over `gen`.
    Returns (node, depth, completions) where completions = [(end_hids, length), ...].
    After a completion the state resets to root (greedy non-overlapping)."""
    node, depth, floor = trie.root, 0, 0
    completions = []
    for i, t in enumerate(gen):
        child = node.children.get(t)
        if child is not None:
            node, depth = child, depth + 1
        else:
            node, depth = trie.root, 0
            for k in range(min(trie.max_depth, i - floor + 1), 0, -1):
                cand = trie.root
                for tt in gen[i - k + 1: i + 1]:
                    cand = cand.children.get(tt)
                    if cand is None:
                        break
                else:
                    node, depth = cand, k
                    break
        if node.end_hids:
            completions.append((frozenset(node.end_hids), depth))
            node, depth, floor = trie.root, 0, i + 1
    return node, depth, completions


class TrieSonarBiasProcessorV2(LogitsProcessor):
    """First-token asymmetry (eta) + failure-arc revocation via potential shaping."""

    def __init__(self, trie, E_bias, W, capture, delta, lam, n_prefix, eta=0.25):
        self.trie, self.E = trie, E_bias
        self.W = W.to(E_bias.device)
        self.capture = capture
        self.delta, self.lam, self.eta = float(delta), float(lam), float(eta)
        self.n_prefix = n_prefix
        self.stats = {"steps": 0, "boosted_steps": 0, "revocation_steps": 0}

    def _epartial(self, d):
        return 0.0 if d <= 0 else self.eta * self.delta + self.delta * (d - 1)

    def _sem(self, sims, b, hids):
        if sims is None or not hids:
            return 0.0
        return self.lam * max(0.0, max(sims[b, h].item() for h in hids))

    def __call__(self, input_ids, scores):
        self.stats["steps"] += 1
        beams = input_ids.shape[0]
        sims = None
        if self.lam > 0 and self.capture.hidden is not None:
            hs = self.capture.hidden
            if hs.shape[0] == beams and hs.shape[1] > self.n_prefix:
                pooled = F.normalize(hs[:, self.n_prefix:, :].mean(dim=1), dim=-1)
                proj = F.normalize(pooled.float() @ self.W, dim=-1)
                sims = (proj @ self.E.T).cpu()

        for b in range(beams):
            gen = input_ids[b, self.n_prefix:].tolist()
            node0, d0, _ = scan_trie(self.trie, gen)
            E0 = self._epartial(d0)
            add = {}
            # Fresh starts: every root child may begin a keyword at eta*delta.
            for t, child in self.trie.root.children.items():
                v = self.eta * self.delta
                if child.end_hids:                      # single-token keyword completes at once
                    v += self._sem(sims, b, child.end_hids)
                if v > add.get(t, 0.0):
                    add[t] = v
            if d0 > 0:
                # Revocation baseline: every token pays back E0 unless it re-earns it below.
                scores[b] -= E0
                self.stats["revocation_steps"] += 1
                for t, child in node0.children.items():
                    if child.end_hids:                  # completing token: lock chain + semantics
                        v = self.eta * self.delta + self.delta * d0 \
                            + self._sem(sims, b, child.end_hids)
                    else:                               # plain continuation: net +delta
                        v = self._epartial(d0 + 1)
                    if v > add.get(t, 0.0):
                        add[t] = v
            if add:
                self.stats["boosted_steps"] += 1
                idx = torch.tensor(list(add.keys()), device=scores.device)
                val = torch.tensor(list(add.values()), device=scores.device,
                                   dtype=scores.dtype)
                scores[b].index_add_(0, idx, val)
        return scores


trie_v2 = TrieV2()
for hid, w in enumerate(bias_words):
    for form in {w, w.capitalize()}:
        for surface in (form, " " + form):
            ids = tok.encode(surface, add_special_tokens=False)
            if ids:
                trie_v2.insert(ids, hid)
print(f"TrieV2 built: {len(bias_words)} words, max_depth={trie_v2.max_depth}")

# Unit checks: marginal arithmetic on a synthetic 3-token keyword path.
_probe = tok.encode(" " + bias_words[0], add_special_tokens=False)
if len(_probe) >= 2:
    n, d, c = scan_trie(trie_v2, _probe[:-1])
    assert d == len(_probe) - 1 and not c, "mid-word scan should hold a partial, no completion"
    n2, d2, c2 = scan_trie(trie_v2, _probe)
    assert c2 and d2 == 0, "full keyword should complete and reset"
    n3, d3, c3 = scan_trie(trie_v2, _probe[:-1] + [999999])
    assert d3 == 0 and not c3, "bogus continuation should fall back with no completion"
    print("scan_trie unit checks passed "
          f"(partial depth {d}, completion length {c2[0][1]})")


---
## Section 8 — Transcription harness + smoke test

`transcribe()`, line-by-line:
1. Audio → log-mel via the feature extractor (pads/trims to Whisper's 30 s window).
2. `use_cache = (lam == 0)` — the crucial switch: A and B keep the KV cache (fast); only C pays the
   full-sequence cost the pooling needs.
3. `whisper.generate(...)` with pinned language/task (no detection pass), configured beam width, and —
   when biasing — our processor appended via `logits_processor` (HF merges it after its own built-in
   processors, so we modify final logits).
4. Decode skipping special tokens.

The smoke test picks the first eval utterance whose reference contains a target word and prints
A/B/C transcripts side by side, plus the processor's fired-step counters — proof the machinery engages
before we spend minutes on the sweep.


In [ ]:
@torch.no_grad()
def transcribe(audio, delta=0.0, lam=0.0):
    feats = processor.feature_extractor(
        audio, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
    kwargs = dict(language="en", task="transcribe",
                  num_beams=CONFIG["beams"], use_cache=(lam == 0.0))
    proc = None
    if delta > 0 or lam > 0:
        capture.hidden = None
        proc = TrieSonarBiasProcessor(trie, E_bias, W, capture, delta, lam, N_PREFIX)
        kwargs["logits_processor"] = LogitsProcessorList([proc])
    ids = whisper.generate(feats, **kwargs)
    text = tok.decode(ids[0], skip_special_tokens=True).strip()
    return text, proc

smoke = next(s for s in eval_set if TARGETSET & set(norm(s["text"]).split()))
present = sorted(TARGETSET & set(norm(smoke["text"]).split()))
audio = get_audio(smoke)
print("REF :", smoke["text"])
print("target words in ref:", present, "\n")
for label, d, l in [("A baseline", 0, 0), ("B trie δ=2", 2, 0), ("C trie+sonar δ=2 λ=4", 2, 4)]:
    hyp, p = transcribe(audio, d, l)
    fired = f"  [boosted {p.stats['boosted_steps']}/{p.stats['steps']} steps]" if p else ""
    print(f"{label:22s}: {hyp}{fired}")


### Section 8b — Worked example: watching the processor think, step by step

The smoke test shows *that* biasing changes the transcript; this cell shows *how*, at token
resolution — the demo to walk the research lead through. A tracing subclass of
`TrieSonarBiasProcessor` records, for **beam 0** (the current top hypothesis; beams reorder between
steps) at every decoding step:

- `hyp_tail` — the last few decoded tokens of the hypothesis so far;
- `deepest_match` — the longest non-empty trie suffix matched (0 = only the root/empty suffix, i.e.
  "a new biasing word *could* start here"; ≥1 = the trie has **locked onto** a partially emitted
  biasing word);
- `boosts` — the top boosted candidate next-tokens with their bonus values. Note the arithmetic:
  every value is `δ + λ·max(0, cos)`, so with δ=2, λ=4 a bonus of ~2.0 means "trie match, but the
  semantic gate sees ≈0 relevance", while ~3.2 means the gate contributed λ·0.3;
- `top_sims` — the three biasing words the semantic pass currently rates most relevant to this
  beam's meaning.

Two things to point out when presenting the table:
1. **Boosts fire at almost every step** even with no in-word match — those are the root's children
  (word-initial tokens of all list words). That ever-present floor is exactly the "word-start
  distortion field" that made δ=3 dangerous in the error-driven evaluation — visible here as data,
  not anecdote.
2. **The in-word rows (deepest_match ≥ 1)** are the mechanism earning its keep: suffix locked,
  bonuses concentrated on the continuation tokens of specific keywords, and (when the gate has
  signal) the target keyword's similarity standing above the distractors in `top_sims`.


In [ ]:
class TracingTrieSonarBiasProcessor(TrieSonarBiasProcessor):
    """Same math as the parent, plus a per-step trace of beam 0."""
    def __init__(self, *args, **kw):
        super().__init__(*args, **kw)
        self.trace = []

    def __call__(self, input_ids, scores):
        self.stats["steps"] += 1
        beams = input_ids.shape[0]
        sims = None
        if self.lam > 0 and self.capture.hidden is not None:
            hs = self.capture.hidden
            if hs.shape[0] == beams and hs.shape[1] > self.n_prefix:
                pooled = F.normalize(hs[:, self.n_prefix:, :].mean(dim=1), dim=-1)
                proj = F.normalize(pooled.float() @ self.W, dim=-1)
                sims = (proj @ self.E.T).cpu()
        for b in range(beams):
            gen = input_ids[b, self.n_prefix:].tolist()
            bonus, matches = {}, []
            lo = max(0, len(gen) - self.trie.max_depth + 1)
            for start in range(lo, len(gen) + 1):
                node = self.trie.walk(gen[start:])
                if node is None:
                    continue
                if b == 0:
                    matches.append((len(gen) - start, gen[start:]))
                for t, child in node.children.items():
                    val = self.delta
                    if sims is not None and child.hids:
                        s = max(sims[b, h].item() for h in child.hids)
                        val += self.lam * max(0.0, s)
                    if val > bonus.get(t, 0.0):
                        bonus[t] = val
            if bonus:
                self.stats["boosted_steps"] += 1
                idx = torch.tensor(list(bonus.keys()), device=scores.device)
                val = torch.tensor(list(bonus.values()), device=scores.device,
                                   dtype=scores.dtype)
                scores[b].index_add_(0, idx, val)
            if b == 0:
                top = sorted(bonus.items(), key=lambda kv: -kv[1])[:4]
                self.trace.append({
                    "step": self.stats["steps"],
                    "hyp_tail": tok.decode(gen[-8:]) if gen else "",
                    "deepest_match": max((d for d, _ in matches), default=0),
                    "in_word_suffix": [repr(tok.decode(seq)) for d, seq in matches if d > 0],
                    "boosts": [(repr(tok.decode([t])), round(v, 2)) for t, v in top],
                    "top_sims": ([(bias_words[i], round(sims[0, i].item(), 3))
                                  for i in sims[0].topk(3).indices.tolist()]
                                 if sims is not None else None),
                })
        return scores

# Trace the same utterance the smoke test used, under condition C settings.
capture.hidden = None
tracer = TracingTrieSonarBiasProcessor(trie, E_bias, W, capture,
                                       delta=2.0, lam=4.0, n_prefix=N_PREFIX)
feats = processor.feature_extractor(
    audio, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
with torch.no_grad():
    out_ids = whisper.generate(feats, language="en", task="transcribe",
                               num_beams=CONFIG["beams"], use_cache=False,
                               logits_processor=LogitsProcessorList([tracer]))

print("REF :", smoke["text"])
print("OUT :", tok.decode(out_ids[0], skip_special_tokens=True), "\n")

trace_df = pd.DataFrame(tracer.trace)
locked = trace_df[trace_df["deepest_match"] > 0]
print(f"{len(trace_df)} steps traced; boosts fired on "
      f"{tracer.stats['boosted_steps']} steps; {len(locked)} steps with an "
      f"in-word trie lock-on (deepest_match ≥ 1)\n")

print("— first 15 steps (root-level boosting = the ever-present word-start floor) —")
display(trace_df.head(15)[["step", "hyp_tail", "deepest_match", "boosts", "top_sims"]])

if len(locked):
    print("— lock-on steps: the trie mid-word, bonuses on continuation tokens —")
    display(locked[["step", "hyp_tail", "in_word_suffix", "boosts", "top_sims"]].head(10))
    r = locked.iloc[0]
    piece, val = r["boosts"][0]
    print(f"\nArithmetic zoom (step {r['step']}): suffix {r['in_word_suffix']} matched; "
          f"top bonus {piece} = δ({tracer.delta}) + λ({tracer.lam})·max(0, cos) = {val}. "
          f"Current top similarities: {r['top_sims']}")
else:
    print("No in-word lock-on occurred on this utterance — rerun with an utterance whose "
          "reference contains a multi-token biasing word (see `missed` selection in the "
          "smoke test).")


### Section 8c — V1 vs V2 on the smoke utterance (+ a directional A/B)

Same utterance, same δ, three decodes: **V1 trie-only** (the configuration whose failure modes
Exp-03 documented), **V2 trie-only** (η=0.25, revocation on — isolates the mechanical fixes), and
**V2 full** (semantic completion bonus on). What to look for:

- V1's characteristic damage — mangled neighbors and orphaned fragments around boosted regions —
  should be absent from V2 outputs: abandoned excursions now net zero, so the beam returns to the
  acoustically honest path.
- `revocation_steps` in the stats is the mechanism visibly working: how often some beam held a
  partial that had to be defended or paid back.
- The small A/B loop (a dozen eval utterances, WER + naive target-presence) is **directional
  only** — the real verdict needs the error-driven notebook rerun with V2 under the spec's
  U-WER-constrained sweep. Expectation from the design: V2 at δ=3 should sit far closer to baseline
  WER than V1 at δ=3, while keeping most of the recall that continuation-strength δ buys.


In [ ]:
print("REF :", smoke["text"])
print("targets in ref:", sorted(set(norm(smoke["text"]).split()) & TARGETSET), "\n")

runs = []
for label, proc_ctor in [
    ("V1 trie-only δ=3",        lambda: TrieSonarBiasProcessor(trie, E_bias, W, capture,
                                                               delta=3.0, lam=0.0,
                                                               n_prefix=N_PREFIX)),
    ("V2 trie-only δ=3 η=0.25", lambda: TrieSonarBiasProcessorV2(trie_v2, E_bias, W, capture,
                                                                 delta=3.0, lam=0.0,
                                                                 n_prefix=N_PREFIX, eta=0.25)),
    ("V2 full δ=3 η=0.25 λ=4",  lambda: TrieSonarBiasProcessorV2(trie_v2, E_bias, W, capture,
                                                                 delta=3.0, lam=4.0,
                                                                 n_prefix=N_PREFIX, eta=0.25)),
]:
    capture.hidden = None
    p = proc_ctor()
    feats = processor.feature_extractor(audio, sampling_rate=16000,
                                        return_tensors="pt").input_features.to(DEVICE)
    with torch.no_grad():
        out_ids = whisper.generate(feats, language="en", task="transcribe",
                                   num_beams=CONFIG["beams"], use_cache=False,
                                   logits_processor=LogitsProcessorList([p]))
    hyp = tok.decode(out_ids[0], skip_special_tokens=True).strip()
    extra = (f"  [boost {p.stats['boosted_steps']}/{p.stats['steps']}"
             + (f", revoked on {p.stats['revocation_steps']} steps"
                if "revocation_steps" in p.stats else "") + "]")
    print(f"{label:26s}: {hyp}{extra}")
    runs.append((label, hyp))

# Directional A/B over a handful of eval utterances (NOT a verdict — see markdown above).
import jiwer
n_ab = min(len(eval_set), 8 if FAST else 12)
sub = eval_set[:n_ab]
refs = [norm(s["text"]) for s in sub]
ab_rows = []
for label, delta, lam, use_v2 in [("V1 δ=3 λ=0", 3.0, 0.0, False),
                                  ("V2 δ=3 η=0.25 λ=0", 3.0, 0.0, True),
                                  ("V2 δ=3 η=0.25 λ=4", 3.0, 4.0, True)]:
    hyps = []
    for s in sub:
        capture.hidden = None
        p = (TrieSonarBiasProcessorV2(trie_v2, E_bias, W, capture, delta=delta, lam=lam,
                                      n_prefix=N_PREFIX, eta=0.25) if use_v2 else
             TrieSonarBiasProcessor(trie, E_bias, W, capture, delta=delta, lam=lam,
                                    n_prefix=N_PREFIX))
        feats = processor.feature_extractor(get_audio(s), sampling_rate=16000,
                                            return_tensors="pt").input_features.to(DEVICE)
        with torch.no_grad():
            out_ids = whisper.generate(feats, language="en", task="transcribe",
                                       num_beams=CONFIG["beams"], use_cache=False,
                                       logits_processor=LogitsProcessorList([p]))
        hyps.append(norm(tok.decode(out_ids[0], skip_special_tokens=True)))
    present = sum(1 for r, h in zip(refs, hyps)
                  for w in set(r.split()) & set(bias_words) if w in h.split())
    total = sum(1 for r in refs for w in set(r.split()) if w in set(bias_words))
    ab_rows.append({"config": label, "wer": round(jiwer.wer(refs, hyps), 3),
                    "targets_present": f"{present}/{total}"})
pd.DataFrame(ab_rows)


---
## Section 9 — Metrics from word alignments

`score_corpus()` computes everything from `jiwer.process_words`, which returns per-sentence
edit-alignment chunks (`equal` / `substitute` / `delete` / `insert`). Attribution rules, line-by-line:

- **`equal`** — each correctly aligned reference word that is a *target* counts as a recall **hit**.
- **`substitute`/`delete`** — the error is attributed by the **reference** word: biased word → `b_err`,
  else `u_err`. A substitution whose **hypothesis** side is a biasing word the reference didn't ask for
  is *additionally* a **false alarm** (the booster overwrote something with a list word).
- **`insert`** — attributed by the **hypothesis** word; inserted biasing words are false alarms.
- Denominators: `b_ref` / `u_ref` = biased / unbiased reference word counts →
  `B-WER = b_err / b_ref`, `U-WER = u_err / u_ref`, `recall = hits / target occurrences`.

This is the standard B-WER/U-WER decomposition from the deep-biasing literature, and it is exactly the
lens that separates "biasing helped" (recall ↑, B-WER ↓) from "biasing vandalized the transcript"
(U-WER ↑, false alarms ↑).


In [ ]:
import jiwer

def score_corpus(refs, hyps):
    R, H = [norm(r) for r in refs], [norm(h) for h in hyps]
    out = jiwer.process_words(R, H)
    hits = occ = b_err = u_err = b_ref = u_ref = fa = 0
    for rtxt, htxt, chunks in zip(R, H, out.alignments):
        rw, hw = rtxt.split(), htxt.split()
        for w in rw:
            if w in HOTSET: b_ref += 1
            else:           u_ref += 1
            if w in TARGETSET: occ += 1
        for ch in chunks:
            if ch.type == "equal":
                hits += sum(1 for i in range(ch.ref_start_idx, ch.ref_end_idx)
                            if rw[i] in TARGETSET)
            elif ch.type in ("substitute", "delete"):
                for i in range(ch.ref_start_idx, ch.ref_end_idx):
                    if rw[i] in HOTSET: b_err += 1
                    else:               u_err += 1
                if ch.type == "substitute":
                    fa += sum(1 for j in range(ch.hyp_start_idx, ch.hyp_end_idx)
                              if hw[j] in HOTSET and hw[j] not in rw)
            elif ch.type == "insert":
                for j in range(ch.hyp_start_idx, ch.hyp_end_idx):
                    if hw[j] in HOTSET:
                        b_err += 1; fa += 1
                    else:
                        u_err += 1
    return {"wer": out.wer,
            "b_wer": b_err / b_ref if b_ref else float("nan"),
            "u_wer": u_err / u_ref if u_ref else float("nan"),
            "recall": hits / occ if occ else float("nan"),
            "false_alarms": fa,
            "target_occurrences": occ}

def run_condition(samples, delta, lam, tag=""):
    hyps = []
    for i, s in enumerate(samples):
        hyp, _ = transcribe(get_audio(s), delta, lam)
        hyps.append(hyp)
        if (i + 1) % 10 == 0:
            print(f"  {tag} {i + 1}/{len(samples)}")
    return hyps

print("Metrics defined.")


---
## Section 10 — Hyperparameter sweep on the dev set

Two stages, dev set only (the eval set stays untouched until Section 11):

1. **Sweep δ with λ=0** (condition B). Expect the classic trade-off: recall climbs with δ, then false
   alarms explode. `δ*` = highest recall whose overall WER hasn't degraded past baseline.
2. **Fix δ\*, sweep λ** (condition C). The semantic term should raise recall further and/or push false
   alarms down at the same δ. `λ*` chosen the same way.

Baseline (0, 0) is scored once for reference. Everything lands in `dev_df` for eyeballing.


In [ ]:
dev_refs_l = [s["text"] for s in dev_set]
rows = []

print("dev baseline...")
hyps = run_condition(dev_set, 0.0, 0.0, "A")
rows.append({"cond": "A", "delta": 0.0, "lam": 0.0, **score_corpus(dev_refs_l, hyps)})

for d in CONFIG["delta_grid"]:
    print(f"dev trie-only δ={d} ...")
    hyps = run_condition(dev_set, d, 0.0, f"B δ={d}")
    rows.append({"cond": "B", "delta": d, "lam": 0.0, **score_corpus(dev_refs_l, hyps)})

dev_df = pd.DataFrame(rows)
base_wer = dev_df.loc[dev_df["cond"] == "A", "wer"].iloc[0]
cand = dev_df[(dev_df["cond"] == "B") & (dev_df["wer"] <= base_wer * 1.05)]
DELTA = float((cand if len(cand) else dev_df[dev_df["cond"] == "B"])
              .sort_values(["recall", "wer"], ascending=[False, True]).iloc[0]["delta"])
print(f"chosen δ* = {DELTA}")

for l in CONFIG["lambda_grid"]:
    print(f"dev trie+sonar δ={DELTA} λ={l} ...")
    hyps = run_condition(dev_set, DELTA, l, f"C λ={l}")
    rows.append({"cond": "C", "delta": DELTA, "lam": l, **score_corpus(dev_refs_l, hyps)})

dev_df = pd.DataFrame(rows)
cand = dev_df[(dev_df["cond"] == "C") & (dev_df["wer"] <= base_wer * 1.05)]
LAM = float((cand if len(cand) else dev_df[dev_df["cond"] == "C"])
            .sort_values(["recall", "wer"], ascending=[False, True]).iloc[0]["lam"])
print(f"chosen λ* = {LAM}")
dev_df.round(3)


---
## Section 11 — Final evaluation (untouched eval set)

The three conditions, tuned once on dev, now scored on the eval utterances. This is the table that
answers the notebook's question. Per-utterance transcripts are also kept for error inspection —
reading a handful of B-vs-C diffs is worth more than any aggregate.


In [ ]:
eval_refs_l = [s["text"] for s in eval_set]
final_rows, transcripts = [], {}

for label, d, l in [("A baseline", 0.0, 0.0),
                    (f"B trie δ={DELTA}", DELTA, 0.0),
                    (f"C trie+SONAR δ={DELTA} λ={LAM}", DELTA, LAM)]:
    print(f"eval: {label} ...")
    hyps = run_condition(eval_set, d, l, label)
    transcripts[label] = hyps
    final_rows.append({"condition": label, **score_corpus(eval_refs_l, hyps)})

final_df = pd.DataFrame(final_rows).set_index("condition")
final_df.round(3)


### Visual summary

Left: WER / B-WER / U-WER per condition — biasing should crush **B-WER** while leaving **U-WER**
untouched. Right: recall (bars) against false alarms (line) — the SONAR condition earns its keep only
if its bar rises **without** its point on the line rising above B's.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

final_df[["wer", "b_wer", "u_wer"]].plot.bar(ax=axes[0], rot=15)
axes[0].set_ylabel("error rate"); axes[0].set_title("WER decomposition")

ax2 = axes[1]
final_df["recall"].plot.bar(ax=ax2, color="tab:green", rot=15)
ax2.set_ylabel("target recall", color="tab:green")
ax3 = ax2.twinx()
ax3.plot(range(len(final_df)), final_df["false_alarms"], "ro-", label="false alarms")
ax3.set_ylabel("false alarms (count)", color="tab:red")
ax2.set_title("Recall vs false alarms")
plt.tight_layout(); plt.show()

diff = [(r, a, b) for r, a, b in zip(eval_refs_l,
                                     transcripts[final_df.index[1]],
                                     transcripts[final_df.index[2]]) if norm(a) != norm(b)]
print(f"\n{len(diff)} utterances differ between B and C; first examples:")
for r, a, b in diff[:3]:
    print("\nREF:", r, "\n B :", a, "\n C :", b)


---
## Section 12 — Save artifacts


In [ ]:
dev_df.to_csv("biasing_dev_sweep.csv", index=False)
final_df.to_csv("biasing_final_results.csv")
pd.DataFrame({"ref": eval_refs_l, **{k: v for k, v in transcripts.items()}}) \
  .to_csv("biasing_eval_transcripts.csv", index=False)
with open("biasing_meta.json", "w") as f:
    json.dump({"config": CONFIG, "delta_star": DELTA, "lambda_star": LAM,
               "dataset": DATA_NAME, "n_targets": len(targets),
               "n_distractors": len(distractors)}, f, indent=2)
hook_handle.remove()
print("Saved: biasing_dev_sweep.csv, biasing_final_results.csv, "
      "biasing_eval_transcripts.csv, biasing_meta.json")


---
## Section 13 — How to read the results

**The verdicts, in decreasing order of joy:**

1. **C > B > A on recall/B-WER, C ≤ B on false alarms, U-WER flat** — the full thesis holds: the trie
   delivers recall and the SONAR gate delivers precision. Ship the architecture; next stop is scale
   (below).
2. **B > A but C ≈ B** — trie biasing works (already a solid win over baseline), but the semantic gate
   adds nothing measurable at this scale. Most likely causes, in order: single-word SONAR embeddings
   are too weak a context signal (try embedding short phrases or topic sentences instead); λ mis-scaled
   (cosines live in a narrow band — try score *normalization* across the list, e.g. softmax, instead
   of raw cosine); eval set too small to resolve the difference.
3. **B ≈ A** — with `whisper-base` and easy LibriSpeech audio, targets may already be recognized
   (recall at baseline near 1.0) leaving no headroom. Check `target_occurrences` and baseline recall
   first; if headroom was absent, rerun with harder audio (`test.other`) or rarer targets rather than
   concluding biasing failed.
4. **Any condition with U-WER or WER clearly above baseline** — δ/λ are over-tuned; the booster is
   vandalizing unbiased words. Lower δ before anything else.

**Known limitations of this implementation (deliberate scope cuts):**
- **KV cache disabled for condition C** — the honest-but-slow way to get full-sequence states. The
  production version keeps the cache and maintains *incremental* per-beam pooled states, reindexed
  each step by `beam_idx` (HF exposes this via `process`/`beam_indices` internals, or port to
  faster-whisper/CTranslate2).
- **No failure/subtractive arcs** — a partially-boosted path that dies keeps its earned bonus
  (Aho-Corasick failure pointers with score revocation are the classical fix; they matter more as δ
  grows and the list gets longer).
- **Word-level SONAR context** — the gate compares the utterance state to *each word alone*. Embedding
  the biasing entries as phrases-in-context, or comparing against a single *list-topic* embedding, are
  natural upgrades the same `W` supports.
- **Scale** — a few dozen utterances bound the confidence here; the same harness runs unchanged on
  hundreds (raise `n_eval`, use CUDA).

**Next steps if the thesis holds:** larger eval (500+ utts, `test.other`), `whisper-small` +
its own refit `W`, phrase-level biasing entries (multi-word names — the trie already supports them),
then the streaming/production port with incremental pooling.
